# Food harvest: starter notebook

Loads the food harvest (`mentions.parquet`, `works.csv`; see `README.md` for columns and caveats),
shows a concordance, and computes first collocation measures. pandas only.

Two things to keep in mind throughout: the data is a **sample** (normalise per work or genre), and the
selection favoured food words near food **verbs**, so compare subsets *within* the harvest
(e.g. `frame` vs `no_frame`, CURE vs COOKING_CREATION) rather than against general Dutch.

In [ ]:
import math, re
from collections import Counter
from pathlib import Path
import pandas as pd

HERE = Path(".")                      # folder with mentions.parquet
df = pd.read_parquet(HERE / "mentions.parquet")
works = pd.read_csv(HERE / "works.csv")
print(len(df), "mentions,", len(works), "works")
df["status"].value_counts()

## Overview: outcome and frames per genre (shares, so genres of different size compare)

In [ ]:
pd.crosstab(df["genre"], df["status"], normalize="index").round(2)

In [ ]:
frames = df[df["status"] == "frame"]
pd.crosstab(frames["genre"], frames["frame"], normalize="index").round(2)

## Concordance

In [ ]:
def conc(data, pref_label=None, frame=None, n=15, width=60, seed=0):
    sub = data
    if pref_label:
        sub = sub[sub["pref_label"] == pref_label]
    if frame:
        sub = sub[sub["frame"] == frame]
    sub = sub.sample(min(n, len(sub)), random_state=seed) if len(sub) > n else sub
    for _, r in sub.iterrows():
        print(f"{str(r.year or ''):>4} {r.left[-width:]:>{width}} [{r.keyword}] {r.right[:width]}")

conc(df, pref_label="suiker", frame="CURE")

## Collocates (log-Dice)

Tokens within `k` words left and right of the keyword. `log-Dice = 14 + log2(2·f(node,coll) / (f(node) + f(coll)))`,
with `f(coll)` counted over all windows in the harvest (the reference). Spelling is pooled crudely by `norm()`;
adjust it to taste. `per_work_cap` limits mentions per work, so one big work (Chomel) cannot dominate.

In [ ]:
TOKEN = re.compile(r"[^\W\d_]+", re.UNICODE)
SPELL = [("uij", "ui"), ("uy", "ui"), ("ij", "y"), ("ck", "k"), ("ae", "a"), ("gh", "g"), ("ph", "f")]

def norm(tok):
    t = tok.lower()
    for a, b in SPELL:
        t = t.replace(a, b)
    return t

def window(row, k=5):
    left = [norm(t) for t in TOKEN.findall(row.left)][-k:]
    right = [norm(t) for t in TOKEN.findall(row.right)][:k]
    return left + right

def cap(data, per_work_cap=50, seed=0):
    return data.groupby("work_id", group_keys=False).apply(
        lambda g: g.sample(min(len(g), per_work_cap), random_state=seed))

reference = Counter(t for w in df.apply(window, axis=1) for t in w)

# Small stop list of (historical) Dutch function words; extend or pass stop=frozenset() to see everything.
STOP = frozenset("""de het een en van in op te met die dat is aan als of voor door men zyn syn zijn
hy hij sy zij ze ik gy gij wy wij u uw hun haar zich niet maar ook so zo dan want om tot by bij na
uyt uit over onder al alle dit dese deze den der des eens wel noch nog seer zeer veel wat wie waer
waar daer daar hier was waren wert wordt worden werd heeft hebben had hadden can kan sal zal souden
zouden moet mogen mag eenen eene ende oft ofte s t""".split())

def collocates(subset, k=5, min_freq=5, top=25, stop=STOP):
    co = Counter(t for w in subset.apply(lambda r: window(r, k), axis=1) for t in w)
    n_node = len(subset)
    rows = [{"collocate": t, "f_co": f, "f_ref": reference[t],
             "log_dice": 14 + math.log2(2 * f / (n_node + reference[t]))}
            for t, f in co.items() if f >= min_freq and t not in stop and len(t) > 2]
    return pd.DataFrame(rows).sort_values("log_dice", ascending=False).head(top)

collocates(cap(df[(df["status"] == "frame") & (df["frame"] == "CURE")]))

## Compare two subsets: CURE vs COOKING_CREATION collocates

In [ ]:
cure = collocates(cap(df[df["frame"] == "CURE"]), top=200).set_index("collocate")["log_dice"]
cook = collocates(cap(df[df["frame"] == "COOKING_CREATION"]), top=200).set_index("collocate")["log_dice"]
both = pd.concat({"CURE": cure, "COOKING_CREATION": cook}, axis=1)
both["diff"] = both["CURE"].fillna(0) - both["COOKING_CREATION"].fillna(0)
both.sort_values("diff").iloc[list(range(12)) + list(range(-12, 0))]

## Keyness: what distinguishes frame mentions from non-frame mentions (log-likelihood)

Both groups went through the same selection, so this contrast is fair where a comparison with
general Dutch would not be.

Expect the selection verbs (*drinken*, *eten*, *koken*) at the top: mentions were selected near food
verbs and the frames follow those verbs. That is the harvest's design showing, not a finding.

In [ ]:
def keyness(a, b, k=5, min_freq=10, top=25):
    ca = Counter(t for w in a.apply(lambda r: window(r, k), axis=1) for t in w)
    cb = Counter(t for w in b.apply(lambda r: window(r, k), axis=1) for t in w)
    na, nb = sum(ca.values()), sum(cb.values())
    out = []
    for t in set(ca) | set(cb):
        x, y = ca[t], cb[t]
        if x + y < min_freq or len(t) <= 2 or t in STOP:
            continue
        ea, eb = na * (x + y) / (na + nb), nb * (x + y) / (na + nb)
        ll = 2 * sum(o * math.log(o / e) for o, e in ((x, ea), (y, eb)) if o)
        out.append({"token": t, "in_a": x, "in_b": y, "LL": ll if x / na > y / nb else -ll})
    return pd.DataFrame(out).sort_values("LL", ascending=False).head(top)

keyness(cap(df[df["status"] == "frame"]), cap(df[df["status"] == "no_frame"]))

## Over time: share of mentions per decade that carry a frame, per genre (works with any mention as denominator)

In [ ]:
t = df[df["decade"].between(1550, 1920)]
share = t.groupby(["decade", "genre"])["status"].apply(lambda s: (s == "frame").mean()).unstack()
counts = t.groupby(["decade", "genre"])["work_id"].nunique().unstack()
share.where(counts >= 3).round(2)   # hide decade x genre cells resting on fewer than 3 works